In [1]:
"""
Harsányi-módszerű performance attribution 100 részvényes globális portfólión.

Bemeneti Excel:
    adatok_python.xlsx (előállítja: adatok_atalakitas.py az ../adatok.xlsx-ből)

Elvárt munkalapok:
    Prices
    Market_Cap
    Metadata

A modell:
    - aktív portfólió: egyenlő súlyozás
    - benchmark: időszakonként újrasúlyozott market-cap súlyozás
    - döntéshozók: REGION, SECTOR, SELECTION
    - értékfüggvény: Davies–Laker (2001) hibrid-portfólió módszer, vagyis a koalíciós
      portfólió és a benchmark kumulált hozamának különbsége
      (Acerbi–Csóka–Herings, 2026, 7.2. szakasz)
    - Harsányi-attribúció
    - Shapley-értékek
    - teljes időszak + részidőszakok
"""

from __future__ import annotations

import itertools
import os
from dataclasses import dataclass
from typing import Dict, FrozenSet, Iterable, List, Tuple

import numpy as np
import pandas as pd


EXCEL_INPUT_PATH = "adatok_python.xlsx"

PRICE_SHEET = "Prices"
MARKET_CAP_SHEET = "Market_Cap"
METADATA_SHEET = "Metadata"

START_DATE = "2016-01-01"
PRICE_START_DATE = "2015-12-31"
END_DATE = "2025-12-31"

MARKET_CAP_REFERENCE_DATE = "2015-12-31"
PERIOD_BENCHMARK_REFERENCE_DATES = {
    "full_2016_2025": "2015-12-31",
    "pre_covid_2016_2019": "2015-12-31",
    "covid_2020_2021": "2019-12-31",
    "inflation_ai_2022_2025": "2021-12-31",
}

PERIODS_PER_YEAR = 12
BASE_CURRENCY = "USD"

DECISION_MAKERS = ["REGION", "SECTOR", "SELECTION"]

OUTPUT_DIR = "outputs_harsanyi_100stocks_hierarchical_davies_laker"
RUN_LABEL = "2016_2025_100stocks_hierarchical_active_marketcap_benchmark_davies_laker"
CREATE_CSV_OUTPUTS = False
DELETE_OLD_EXCEL_OUTPUTS = True


ASSET_METADATA = {
    "MSFT US Equity": {"company": "Microsoft", "country": "United States", "region": "North America", "sector": "Information Technology"},
    "AAPL US Equity": {"company": "Apple", "country": "United States", "region": "North America", "sector": "Information Technology"},
    "NVDA US Equity": {"company": "Nvidia", "country": "United States", "region": "North America", "sector": "Information Technology"},
    "JPM US Equity": {"company": "JPMorgan Chase", "country": "United States", "region": "North America", "sector": "Financials"},
    "LLY US Equity": {"company": "Eli Lilly", "country": "United States", "region": "North America", "sector": "Health Care"},
    "JNJ US Equity": {"company": "Johnson & Johnson", "country": "United States", "region": "North America", "sector": "Health Care"},
    "XOM US Equity": {"company": "Exxon Mobil", "country": "United States", "region": "North America", "sector": "Energy"},
    "PG US Equity": {"company": "Procter & Gamble", "country": "United States", "region": "North America", "sector": "Consumer Staples"},
    "ASML NA Equity": {"company": "ASML", "country": "Netherlands", "region": "Europe", "sector": "Information Technology"},
    "SAP GY Equity": {"company": "SAP", "country": "Germany", "region": "Europe", "sector": "Information Technology"},
    "NESN SW Equity": {"company": "Nestlé", "country": "Switzerland", "region": "Europe", "sector": "Consumer Staples"},
    "NOVOB DC Equity": {"company": "Novo Nordisk", "country": "Denmark", "region": "Europe", "sector": "Health Care"},
    "SHEL LN Equity": {"company": "Shell", "country": "United Kingdom", "region": "Europe", "sector": "Energy"},
    "MC FP Equity": {"company": "LVMH", "country": "France", "region": "Europe", "sector": "Consumer Discretionary"},
    "SIE GY Equity": {"company": "Siemens", "country": "Germany", "region": "Europe", "sector": "Industrials"},
    "7203 JT Equity": {"company": "Toyota", "country": "Japan", "region": "Asia", "sector": "Consumer Discretionary"},
    "6758 JT Equity": {"company": "Sony", "country": "Japan", "region": "Asia", "sector": "Consumer Discretionary"},
    "005930 KS Equity": {"company": "Samsung Electronics", "country": "South Korea", "region": "Asia", "sector": "Information Technology"},
    "2330 TT Equity": {"company": "TSMC", "country": "Taiwan", "region": "Asia", "sector": "Information Technology"},
    "8306 JT Equity": {"company": "Mitsubishi UFJ Financial Group", "country": "Japan", "region": "Asia", "sector": "Financials"},
    "AVGO US Equity": {"company": "Broadcom", "country": "United States", "region": "North America", "sector": "Information Technology"},
    "IFX GY Equity": {"company": "Infineon Technologies", "country": "Germany", "region": "Europe", "sector": "Information Technology"},
    "BRK/B US Equity": {"company": "Berkshire Hathaway", "country": "United States", "region": "North America", "sector": "Financials"},
    "BAC US Equity": {"company": "Bank of America", "country": "United States", "region": "North America", "sector": "Financials"},
    "GS US Equity": {"company": "Goldman Sachs", "country": "United States", "region": "North America", "sector": "Financials"},
    "HSBA LN Equity": {"company": "HSBC", "country": "United Kingdom", "region": "Europe", "sector": "Financials"},
    "ALV GY Equity": {"company": "Allianz", "country": "Germany", "region": "Europe", "sector": "Financials"},
    "8316 JT Equity": {"company": "Sumitomo Mitsui Financial Group", "country": "Japan", "region": "Asia", "sector": "Financials"},
    "DBS SP Equity": {"company": "DBS Group", "country": "Singapore", "region": "Asia", "sector": "Financials"},
    "UNH US Equity": {"company": "UnitedHealth Group", "country": "United States", "region": "North America", "sector": "Health Care"},
    "ABBV US Equity": {"company": "AbbVie", "country": "United States", "region": "North America", "sector": "Health Care"},
    "ROG SW Equity": {"company": "Roche", "country": "Switzerland", "region": "Europe", "sector": "Health Care"},
    "AZN LN Equity": {"company": "AstraZeneca", "country": "United Kingdom", "region": "Europe", "sector": "Health Care"},
    "4502 JT Equity": {"company": "Takeda Pharmaceutical", "country": "Japan", "region": "Asia", "sector": "Health Care"},
    "4568 JT Equity": {"company": "Daiichi Sankyo", "country": "Japan", "region": "Asia", "sector": "Health Care"},
    "CVX US Equity": {"company": "Chevron", "country": "United States", "region": "North America", "sector": "Energy"},
    "COP US Equity": {"company": "ConocoPhillips", "country": "United States", "region": "North America", "sector": "Energy"},
    "EOG US Equity": {"company": "EOG Resources", "country": "United States", "region": "North America", "sector": "Energy"},
    "TTE FP Equity": {"company": "TotalEnergies", "country": "France", "region": "Europe", "sector": "Energy"},
    "EQNR NO Equity": {"company": "Equinor", "country": "Norway", "region": "Europe", "sector": "Energy"},
    "857 HK Equity": {"company": "PetroChina", "country": "China", "region": "Asia", "sector": "Energy"},
    "386 HK Equity": {"company": "Sinopec", "country": "China", "region": "Asia", "sector": "Energy"},
    "WMT US Equity": {"company": "Walmart", "country": "United States", "region": "North America", "sector": "Consumer Staples"},
    "COST US Equity": {"company": "Costco Wholesale", "country": "United States", "region": "North America", "sector": "Consumer Staples"},
    "KO US Equity": {"company": "Coca-Cola", "country": "United States", "region": "North America", "sector": "Consumer Staples"},
    "ULVR LN Equity": {"company": "Unilever", "country": "United Kingdom", "region": "Europe", "sector": "Consumer Staples"},
    "DGE LN Equity": {"company": "Diageo", "country": "United Kingdom", "region": "Europe", "sector": "Consumer Staples"},
    "600519 CH Equity": {"company": "Kweichow Moutai", "country": "China", "region": "Asia", "sector": "Consumer Staples"},
    "2914 JT Equity": {"company": "Japan Tobacco", "country": "Japan", "region": "Asia", "sector": "Consumer Staples"},
    "AMZN US Equity": {"company": "Amazon", "country": "United States", "region": "North America", "sector": "Consumer Discretionary"},
    "HD US Equity": {"company": "Home Depot", "country": "United States", "region": "North America", "sector": "Consumer Discretionary"},
    "MCD US Equity": {"company": "McDonald's", "country": "United States", "region": "North America", "sector": "Consumer Discretionary"},
    "BMW GY Equity": {"company": "BMW", "country": "Germany", "region": "Europe", "sector": "Consumer Discretionary"},
    "RMS FP Equity": {"company": "Hermès", "country": "France", "region": "Europe", "sector": "Consumer Discretionary"},
    "1211 HK Equity": {"company": "BYD", "country": "China", "region": "Asia", "sector": "Consumer Discretionary"},
    "CAT US Equity": {"company": "Caterpillar", "country": "United States", "region": "North America", "sector": "Industrials"},
    "GE US Equity": {"company": "GE Aerospace", "country": "United States", "region": "North America", "sector": "Industrials"},
    "HON US Equity": {"company": "Honeywell", "country": "United States", "region": "North America", "sector": "Industrials"},
    "SU FP Equity": {"company": "Schneider Electric", "country": "France", "region": "Europe", "sector": "Industrials"},
    "AIR FP Equity": {"company": "Airbus", "country": "France", "region": "Europe", "sector": "Industrials"},
    "7011 JT Equity": {"company": "Mitsubishi Heavy Industries", "country": "Japan", "region": "Asia", "sector": "Industrials"},
    "6501 JT Equity": {"company": "Hitachi", "country": "Japan", "region": "Asia", "sector": "Industrials"},
    "6301 JT Equity": {"company": "Komatsu", "country": "Japan", "region": "Asia", "sector": "Industrials"},
    "GOOGL US Equity": {"company": "Alphabet", "country": "United States", "region": "North America", "sector": "Communication Services"},
    "META US Equity": {"company": "Meta Platforms", "country": "United States", "region": "North America", "sector": "Communication Services"},
    "NFLX US Equity": {"company": "Netflix", "country": "United States", "region": "North America", "sector": "Communication Services"},
    "VZ US Equity": {"company": "Verizon", "country": "United States", "region": "North America", "sector": "Communication Services"},
    "TMUS US Equity": {"company": "T-Mobile US", "country": "United States", "region": "North America", "sector": "Communication Services"},
    "DTE GY Equity": {"company": "Deutsche Telekom", "country": "Germany", "region": "Europe", "sector": "Communication Services"},
    "VOD LN Equity": {"company": "Vodafone", "country": "United Kingdom", "region": "Europe", "sector": "Communication Services"},
    "700 HK Equity": {"company": "Tencent", "country": "China", "region": "Asia", "sector": "Communication Services"},
    "9432 JT Equity": {"company": "NTT", "country": "Japan", "region": "Asia", "sector": "Communication Services"},
    "LIN US Equity": {"company": "Linde", "country": "United States", "region": "North America", "sector": "Materials"},
    "FCX US Equity": {"company": "Freeport-McMoRan", "country": "United States", "region": "North America", "sector": "Materials"},
    "NEM US Equity": {"company": "Newmont", "country": "United States", "region": "North America", "sector": "Materials"},
    "RIO LN Equity": {"company": "Rio Tinto", "country": "United Kingdom", "region": "Europe", "sector": "Materials"},
    "AI FP Equity": {"company": "Air Liquide", "country": "France", "region": "Europe", "sector": "Materials"},
    "GLEN LN Equity": {"company": "Glencore", "country": "United Kingdom", "region": "Europe", "sector": "Materials"},
    "BHP AU Equity": {"company": "BHP", "country": "Australia", "region": "Asia", "sector": "Materials"},
    "4063 JT Equity": {"company": "Shin-Etsu Chemical", "country": "Japan", "region": "Asia", "sector": "Materials"},
    "005490 KS Equity": {"company": "POSCO Holdings", "country": "South Korea", "region": "Asia", "sector": "Materials"},
    "PLD US Equity": {"company": "Prologis", "country": "United States", "region": "North America", "sector": "Real Estate"},
    "EQIX US Equity": {"company": "Equinix", "country": "United States", "region": "North America", "sector": "Real Estate"},
    "VNA GY Equity": {"company": "Vonovia", "country": "Germany", "region": "Europe", "sector": "Real Estate"},
    "SGRO LN Equity": {"company": "Segro", "country": "United Kingdom", "region": "Europe", "sector": "Real Estate"},
    "LAND LN Equity": {"company": "Land Securities", "country": "United Kingdom", "region": "Europe", "sector": "Real Estate"},
    "8801 JT Equity": {"company": "Mitsui Fudosan", "country": "Japan", "region": "Asia", "sector": "Real Estate"},
    "8802 JT Equity": {"company": "Mitsubishi Estate", "country": "Japan", "region": "Asia", "sector": "Real Estate"},
    "16 HK Equity": {"company": "Sun Hung Kai Properties", "country": "Hong Kong", "region": "Asia", "sector": "Real Estate"},
    "823 HK Equity": {"company": "Link REIT", "country": "Hong Kong", "region": "Asia", "sector": "Real Estate"},
    "NEE US Equity": {"company": "NextEra Energy", "country": "United States", "region": "North America", "sector": "Utilities"},
    "DUK US Equity": {"company": "Duke Energy", "country": "United States", "region": "North America", "sector": "Utilities"},
    "SO US Equity": {"company": "Southern Company", "country": "United States", "region": "North America", "sector": "Utilities"},
    "SRE US Equity": {"company": "Sempra", "country": "United States", "region": "North America", "sector": "Utilities"},
    "IBE SM Equity": {"company": "Iberdrola", "country": "Spain", "region": "Europe", "sector": "Utilities"},
    "ENEL IM Equity": {"company": "Enel", "country": "Italy", "region": "Europe", "sector": "Utilities"},
    "600900 CH Equity": {"company": "China Yangtze Power", "country": "China", "region": "Asia", "sector": "Utilities"},
    "2 HK Equity": {"company": "CLP Holdings", "country": "Hong Kong", "region": "Asia", "sector": "Utilities"},
    "PWGR IN Equity": {"company": "Power Grid Corporation of India", "country": "India", "region": "Asia", "sector": "Utilities"},
    "NTPC IN Equity": {"company": "NTPC", "country": "India", "region": "Asia", "sector": "Utilities"},
}


@dataclass
class PortfolioData:
    prices: pd.DataFrame
    returns: pd.DataFrame
    active_weights: pd.Series
    benchmark_weights: pd.Series
    market_caps: pd.Series
    market_cap_history: pd.DataFrame
    metadata: pd.DataFrame
    benchmark_reference_date: str


@dataclass
class AttributionActions:
    region_weights: pd.Series
    sector_weights: pd.Series
    cell_returns: pd.DataFrame


@dataclass
class AttributionResult:
    period_name: str
    start_date: str
    end_date: str
    summary: pd.DataFrame
    attribution: pd.DataFrame
    shapley: pd.DataFrame
    efficiency_check: pd.DataFrame
    performance_timeseries: pd.DataFrame


def ensure_output_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def delete_old_excel_outputs(output_dir: str) -> None:
    if not os.path.isdir(output_dir):
        return

    for filename in os.listdir(output_dir):
        if filename.lower().endswith(".xlsx"):
            os.remove(os.path.join(output_dir, filename))


def powerset(items: List[str]) -> Iterable[Tuple[str, ...]]:
    for r in range(len(items) + 1):
        for combo in itertools.combinations(items, r):
            yield combo


def normalize_weights(weights: pd.Series) -> pd.Series:
    weights = weights.astype(float)
    total = weights.sum()

    if total <= 0 or np.isnan(total):
        raise ValueError(
            "A súlyok összege nem pozitív, nem lehet normalizálni."
        )

    return weights / total


def cumulative_return(returns: pd.Series) -> pd.Series:
    returns = returns.dropna()
    return (1.0 + returns).cumprod() - 1.0


def annualized_return(
    returns: pd.Series,
    periods_per_year: int = PERIODS_PER_YEAR
) -> float:

    returns = returns.dropna()

    if returns.empty:
        return np.nan

    growth = (1.0 + returns).prod()
    n_periods = len(returns)

    if n_periods <= 0 or growth <= 0:
        return np.nan

    return float(
        growth ** (periods_per_year / n_periods) - 1.0
    )


def annualized_volatility(
    returns: pd.Series,
    periods_per_year: int = PERIODS_PER_YEAR
) -> float:

    returns = returns.dropna()

    if len(returns) <= 1:
        return np.nan

    return float(
        returns.std(ddof=1) * np.sqrt(periods_per_year)
    )


def sharpe_ratio_0rf(
    returns: pd.Series,
    periods_per_year: int = PERIODS_PER_YEAR
) -> float:

    returns = returns.dropna()

    if len(returns) <= 1:
        return np.nan

    vol = returns.std(ddof=1)

    if vol == 0 or np.isnan(vol):
        return np.nan

    return float(
        (returns.mean() / vol) * np.sqrt(periods_per_year)
    )


def max_drawdown(returns: pd.Series) -> float:
    returns = returns.dropna()

    if returns.empty:
        return np.nan

    wealth = (1.0 + returns).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1.0

    return float(drawdown.min())


def summarize_return_series(
    name: str,
    returns: pd.Series
) -> Dict[str, float | str | int]:

    returns = returns.dropna()

    return {
        "name": name,
        "cumulative_return": (
            float((1.0 + returns).prod() - 1.0)
            if not returns.empty
            else np.nan
        ),
        "annualized_return": annualized_return(returns),
        "annualized_volatility": annualized_volatility(returns),
        "sharpe_0rf": sharpe_ratio_0rf(returns),
        "max_drawdown": max_drawdown(returns),
        "mean_monthly_return": (
            float(returns.mean())
            if not returns.empty
            else np.nan
        ),
        "std_monthly_return": (
            float(returns.std(ddof=1))
            if len(returns) > 1
            else np.nan
        ),
        "n_obs": len(returns),
    }


def safe_excel_sheet_name(name: str) -> str:
    invalid_chars = ["[", "]", ":", "*", "?", "/", "\\"]

    safe_name = name

    for char in invalid_chars:
        safe_name = safe_name.replace(char, "_")

    return safe_name[:31]


def load_excel_table(
    excel_path: str,
    sheet_name: str
) -> pd.DataFrame:

    if not os.path.exists(excel_path):
        raise FileNotFoundError(
            f"Nem találom az Excel-fájlt: {excel_path}"
        )

    df = pd.read_excel(
        excel_path,
        sheet_name=sheet_name,
        engine="openpyxl"
    )

    if "Date" not in df.columns:
        raise ValueError(
            f"A(z) '{sheet_name}' munkalapon nincs 'Date' oszlop."
        )

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    df = df.dropna(subset=["Date"])
    df = df.sort_values("Date")
    df = df.set_index("Date")

    return df


def load_metadata_from_excel(
    excel_path: str,
    sheet_name: str
) -> pd.DataFrame:

    metadata_excel = pd.read_excel(
        excel_path,
        sheet_name=sheet_name,
        engine="openpyxl"
    )

    required_columns = {
        "Bloomberg_Ticker",
        "Company"
    }

    missing = required_columns - set(metadata_excel.columns)

    if missing:
        raise ValueError(
            "Hiányzó oszlop(ok) a Metadata munkalapon: "
            + ", ".join(sorted(missing))
        )

    metadata_excel = metadata_excel.dropna(
        subset=["Bloomberg_Ticker"]
    )

    return metadata_excel


def validate_ticker_set(
    price_columns: List[str],
    market_cap_columns: List[str],
    metadata_excel: pd.DataFrame
) -> List[str]:

    price_tickers = list(price_columns)
    market_cap_tickers = list(market_cap_columns)

    if len(price_tickers) != 100:
        raise ValueError(
            f"A Prices munkalapon 100 részvényt várunk, "
            f"de {len(price_tickers)} található."
        )

    if len(market_cap_tickers) != 100:
        raise ValueError(
            f"A Market_Cap munkalapon 100 részvényt várunk, "
            f"de {len(market_cap_tickers)} található."
        )

    if set(price_tickers) != set(market_cap_tickers):
        missing_market_cap = set(price_tickers) - set(market_cap_tickers)
        missing_price = set(market_cap_tickers) - set(price_tickers)

        raise ValueError(
            "A Prices és Market_Cap tickerlistája eltér.\n"
            f"Hiányzik Market_Cap-ból: {sorted(missing_market_cap)}\n"
            f"Hiányzik Prices-ből: {sorted(missing_price)}"
        )

    metadata_tickers = set(
        metadata_excel["Bloomberg_Ticker"].astype(str)
    )

    missing_metadata_excel = (
        set(price_tickers) - metadata_tickers
    )

    if missing_metadata_excel:
        raise ValueError(
            "A következő tickerek hiányoznak a Metadata lapról: "
            + ", ".join(sorted(missing_metadata_excel))
        )

    missing_model_metadata = (
        set(price_tickers) - set(ASSET_METADATA)
    )

    if missing_model_metadata:
        raise ValueError(
            "A következő tickerekhez nincs REGION/SECTOR besorolás: "
            + ", ".join(sorted(missing_model_metadata))
        )

    return price_tickers


def load_prices_and_market_caps(
    excel_path: str,
    start: str,
    end: str
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:

    prices = load_excel_table(
        excel_path,
        PRICE_SHEET
    )

    market_caps = load_excel_table(
        excel_path,
        MARKET_CAP_SHEET
    )

    metadata_excel = load_metadata_from_excel(
        excel_path,
        METADATA_SHEET
    )

    tickers = validate_ticker_set(
        price_columns=list(prices.columns),
        market_cap_columns=list(market_caps.columns),
        metadata_excel=metadata_excel
    )

    prices = prices.loc[start:end, tickers].copy()
    market_caps = market_caps.loc[:, tickers].copy()

    prices = prices.apply(
        pd.to_numeric,
        errors="coerce"
    )

    market_caps = market_caps.apply(
        pd.to_numeric,
        errors="coerce"
    )

    prices = prices.dropna(
        axis=0,
        how="any"
    )

    if prices.empty:
        raise ValueError(
            "A megadott időszakban nincs használható árfolyamadat."
        )

    if prices.isna().any().any():
        raise ValueError(
            "Hiányzó árfolyamérték maradt az adatbázisban."
        )

    ref_date = pd.to_datetime(
        MARKET_CAP_REFERENCE_DATE
    )

    eligible_dates = market_caps.index[
        market_caps.index <= ref_date
    ]

    if len(eligible_dates) == 0:
        raise ValueError(
            "Nincs market cap adat a benchmark referencia-dátuma előtt."
        )

    used_reference_date = eligible_dates.max()

    market_cap_reference = market_caps.loc[
        used_reference_date,
        tickers
    ].copy()

    market_cap_reference = pd.to_numeric(
        market_cap_reference,
        errors="coerce"
    )

    if market_cap_reference.isna().any():
        missing = market_cap_reference[
            market_cap_reference.isna()
        ].index.tolist()

        raise ValueError(
            "Hiányzó market cap a benchmark referencia-dátumán "
            f"({used_reference_date.date()}): {missing}"
        )

    if (market_cap_reference <= 0).any():
        invalid = market_cap_reference[
            market_cap_reference <= 0
        ].index.tolist()

        raise ValueError(
            "Nem pozitív market cap érték: "
            + ", ".join(invalid)
        )

    market_cap_reference.name = "market_cap_usd"

    market_cap_reference.attrs[
        "reference_date_requested"
    ] = MARKET_CAP_REFERENCE_DATE

    market_cap_reference.attrs[
        "reference_date_used"
    ] = str(used_reference_date.date())

    return prices, market_cap_reference, market_caps, tickers


def calculate_simple_returns(
    prices: pd.DataFrame
) -> pd.DataFrame:

    returns = prices.pct_change().dropna(
        how="any"
    )

    if returns.empty:
        raise ValueError(
            "A hozamtábla üres."
        )

    return returns


def build_hierarchical_equal_weights(
    tickers: List[str],
    metadata: pd.DataFrame
) -> pd.Series:
    """
    Build the active portfolio using hierarchical equal weighting:
    - equal total weight across regions;
    - equal sector weight within each region;
    - equal stock weight within each region-sector cell.
    """
    if not tickers:
        raise ValueError("Nincs ticker.")

    missing_metadata = sorted(set(tickers) - set(metadata.index))
    if missing_metadata:
        raise ValueError(f"Hiányzó metadata a tickerekhez: {missing_metadata}")

    meta = metadata.loc[tickers, ["region", "sector"]].copy()
    if meta[["region", "sector"]].isna().any().any():
        raise ValueError("Hiányzó régió vagy szektor a metadata táblában.")

    regions = sorted(meta["region"].unique())
    if not regions:
        raise ValueError("Nincs régió a metadata alapján.")

    weights = pd.Series(0.0, index=tickers, dtype=float, name="active_weight")
    region_weight = 1.0 / len(regions)

    for region in regions:
        region_stocks = meta[meta["region"] == region]
        sectors = sorted(region_stocks["sector"].unique())
        sector_weight = region_weight / len(sectors)

        for sector in sectors:
            cell = region_stocks[region_stocks["sector"] == sector]
            stock_weight = sector_weight / len(cell)
            weights.loc[cell.index] = stock_weight

    if not np.isclose(weights.sum(), 1.0):
        raise ValueError(f"Az aktív súlyok összege nem 1: {weights.sum():.12f}")

    return weights


def build_benchmark_weights_from_market_caps(
    market_cap_history: pd.DataFrame,
    tickers: List[str],
    reference_date: str
) -> Tuple[pd.Series, str]:
    """Create fixed benchmark weights from the latest available market cap on/before reference_date."""
    ref_date = pd.to_datetime(reference_date)
    eligible_dates = market_cap_history.index[market_cap_history.index <= ref_date]
    if len(eligible_dates) == 0:
        raise ValueError(f"Nincs market cap adat a benchmark referencia-dátuma előtt: {reference_date}")
    used_reference_date = eligible_dates.max()
    reference_caps = pd.to_numeric(market_cap_history.loc[used_reference_date, tickers], errors="coerce")
    if reference_caps.isna().any():
        missing = reference_caps[reference_caps.isna()].index.tolist()
        raise ValueError(f"Hiányzó market cap a benchmark referencia-dátumán ({used_reference_date.date()}): {missing}")
    if (reference_caps <= 0).any():
        invalid = reference_caps[reference_caps <= 0].index.tolist()
        raise ValueError(f"Nem pozitív market cap a benchmark referencia-dátumán ({used_reference_date.date()}): {invalid}")
    weights = normalize_weights(reference_caps)
    weights.name = "benchmark_weight"
    weights.attrs["reference_date_requested"] = reference_date
    weights.attrs["reference_date_used"] = str(used_reference_date.date())
    return weights, str(used_reference_date.date())


def load_portfolio_data() -> PortfolioData:

    prices, market_cap_reference, market_cap_history, tickers = (
        load_prices_and_market_caps(
            EXCEL_INPUT_PATH,
            PRICE_START_DATE,
            END_DATE
        )
    )

    returns = calculate_simple_returns(prices)

    metadata_rows = []
    for ticker in tickers:
        metadata_rows.append({
            "ticker": ticker,
            "company": ASSET_METADATA[ticker]["company"],
            "country": ASSET_METADATA[ticker]["country"],
            "region": ASSET_METADATA[ticker]["region"],
            "sector": ASSET_METADATA[ticker]["sector"],
        })

    metadata = pd.DataFrame(metadata_rows).set_index("ticker")

    active_weights = build_hierarchical_equal_weights(
        tickers,
        metadata
    )

    benchmark_weights, benchmark_reference_date = build_benchmark_weights_from_market_caps(
        market_cap_history,
        tickers,
        MARKET_CAP_REFERENCE_DATE
    )

    return PortfolioData(
        prices=prices.loc[returns.index, tickers],
        returns=returns,
        active_weights=active_weights,
        benchmark_weights=benchmark_weights,
        market_caps=market_cap_reference,
        market_cap_history=market_cap_history,
        metadata=metadata,
        benchmark_reference_date=benchmark_reference_date
    )


def portfolio_returns_rebalanced(
    returns: pd.DataFrame,
    weights: pd.Series
) -> pd.Series:

    common_assets = returns.columns.intersection(
        weights.index
    )

    if len(common_assets) == 0:
        raise ValueError(
            "Nincs közös elem a returns és weights között."
        )

    r = returns.loc[
        :,
        common_assets
    ]

    w = normalize_weights(
        weights.loc[common_assets]
    )

    result = r.dot(w)
    result.name = "portfolio_return"

    return result


def build_performance_timeseries(
    data: PortfolioData
) -> pd.DataFrame:

    active_ret = portfolio_returns_rebalanced(
        data.returns,
        data.active_weights
    )

    benchmark_ret = portfolio_returns_rebalanced(
        data.returns,
        data.benchmark_weights
    )

    common_index = (
        active_ret.index
        .intersection(benchmark_ret.index)
    )

    active_ret = active_ret.loc[
        common_index
    ]

    benchmark_ret = benchmark_ret.loc[
        common_index
    ]

    excess_ret = (
        active_ret - benchmark_ret
    )

    return pd.DataFrame(
        {
            "active_return": active_ret,
            "benchmark_return": benchmark_ret,
            "excess_return": excess_ret,
            "cumulative_active_return":
                cumulative_return(active_ret),
            "cumulative_benchmark_return":
                cumulative_return(benchmark_ret),
            "cumulative_excess_return":
                cumulative_return(active_ret)
                - cumulative_return(benchmark_ret),
        }
    )


def summarize_active_vs_benchmark(
    data: PortfolioData
) -> pd.DataFrame:

    ts = build_performance_timeseries(
        data
    )

    active_ret = ts[
        "active_return"
    ]

    benchmark_ret = ts[
        "benchmark_return"
    ]

    excess_ret = ts[
        "excess_return"
    ]

    excess_std = excess_ret.std(
        ddof=1
    )

    rows = [
        summarize_return_series(
            "Active hierarchical equal-weight portfolio",
            active_ret
        ),
        summarize_return_series(
            "Market-cap weighted benchmark",
            benchmark_ret
        ),
        {
            "name": "Excess return",
            "cumulative_excess_return":
                float(
                    (1.0 + active_ret).prod()
                    - (1.0 + benchmark_ret).prod()
                ),
            "monthly_mean":
                float(excess_ret.mean()),
            "monthly_std":
                float(excess_std),
            "tracking_error_annualized":
                float(
                    excess_std
                    * np.sqrt(PERIODS_PER_YEAR)
                ),
            "information_ratio":
                (
                    float(
                        excess_ret.mean()
                        / excess_std
                        * np.sqrt(PERIODS_PER_YEAR)
                    )
                    if excess_std != 0
                    else np.nan
                ),
            "n_obs": len(excess_ret),
        },
    ]

    return pd.DataFrame(
        rows
    ).set_index("name")


def compute_region_weights(
    stock_weights: pd.Series,
    metadata: pd.DataFrame
) -> pd.Series:

    df = metadata.copy()

    df["weight"] = stock_weights.loc[
        df.index
    ]

    region_weights = (
        df.groupby("region")["weight"]
        .sum()
    )

    region_weights = normalize_weights(
        region_weights
    )

    region_weights.name = "region_weight"

    return region_weights


def compute_sector_weights_within_region(
    stock_weights: pd.Series,
    metadata: pd.DataFrame
) -> pd.Series:

    df = metadata.copy()

    df["weight"] = stock_weights.loc[
        df.index
    ]

    cell_weights = (
        df.groupby(
            ["region", "sector"]
        )["weight"]
        .sum()
    )

    region_weights = (
        df.groupby("region")["weight"]
        .sum()
    )

    sector_within_region = (
        cell_weights.copy()
    )

    for region in region_weights.index:

        mask = (
            sector_within_region.index
            .get_level_values("region")
            == region
        )

        sector_within_region.loc[mask] = (
            sector_within_region.loc[mask]
            / region_weights.loc[region]
        )

    sector_within_region.name = (
        "sector_weight_within_region"
    )

    return sector_within_region


def compute_cell_returns(
    returns: pd.DataFrame,
    stock_weights: pd.Series,
    metadata: pd.DataFrame
) -> pd.DataFrame:

    cell_return_dict = {}

    for (
        region,
        sector
    ), group in metadata.groupby(
        ["region", "sector"]
    ):

        tickers_in_cell = list(
            group.index
        )

        cell_stock_weights = (
            normalize_weights(
                stock_weights.loc[
                    tickers_in_cell
                ]
            )
        )

        cell_returns = (
            returns.loc[
                :,
                tickers_in_cell
            ].dot(
                cell_stock_weights
            )
        )

        cell_return_dict[
            (region, sector)
        ] = cell_returns

    cell_returns_df = pd.DataFrame(
        cell_return_dict
    )

    cell_returns_df.columns = (
        pd.MultiIndex.from_tuples(
            cell_returns_df.columns,
            names=["region", "sector"]
        )
    )

    return cell_returns_df


def build_attribution_actions(
    returns: pd.DataFrame,
    stock_weights: pd.Series,
    metadata: pd.DataFrame
) -> AttributionActions:

    return AttributionActions(
        region_weights=compute_region_weights(
            stock_weights,
            metadata
        ),
        sector_weights=compute_sector_weights_within_region(
            stock_weights,
            metadata
        ),
        cell_returns=compute_cell_returns(
            returns,
            stock_weights,
            metadata
        )
    )


def return_function(
    actions: AttributionActions,
    all_cells: pd.MultiIndex
) -> pd.Series:

    dates = actions.cell_returns.index

    result = pd.Series(
        0.0,
        index=dates,
        name="portfolio_return"
    )

    for region, sector in all_cells:

        if region not in actions.region_weights.index:
            continue

        if (
            region,
            sector
        ) not in actions.sector_weights.index:
            continue

        if (
            region,
            sector
        ) not in actions.cell_returns.columns:
            continue

        result = result + (
            actions.region_weights.loc[region]
            * actions.sector_weights.loc[
                (region, sector)
            ]
            * actions.cell_returns.loc[
                :,
                (region, sector)
            ]
        )

    return result


def mix_actions(
    coalition: FrozenSet[str],
    active_actions: AttributionActions,
    benchmark_actions: AttributionActions
) -> AttributionActions:

    return AttributionActions(
        region_weights=(
            active_actions.region_weights
            if "REGION" in coalition
            else benchmark_actions.region_weights
        ),
        sector_weights=(
            active_actions.sector_weights
            if "SECTOR" in coalition
            else benchmark_actions.sector_weights
        ),
        cell_returns=(
            active_actions.cell_returns
            if "SELECTION" in coalition
            else benchmark_actions.cell_returns
        )
    )


def cumulative_excess_return(
    portfolio_returns: pd.Series,
    benchmark_returns: pd.Series
) -> float:
    """
    Davies és Laker (2001) módszere: a több időszakra kumulált hozamot
    közvetlenül a hibrid portfólió havi hozamaiból számoljuk (mértani
    láncolás), nem az egyes hónapok hatásainak összeadásával.

    Az értékfüggvény a portfólió és a benchmark kumulált hozamának különbsége:
        prod(1 + R_P,t) - prod(1 + R_B,t)
    (Acerbi, Csóka és Herings, 2026, 7.2. szakasz.)
    """
    portfolio_growth = (1.0 + portfolio_returns).prod()
    benchmark_growth = (1.0 + benchmark_returns).prod()

    return float(portfolio_growth - benchmark_growth)


def value_function_excess_return(
    coalition: FrozenSet[str],
    active_actions: AttributionActions,
    benchmark_actions: AttributionActions,
    all_cells: pd.MultiIndex
) -> float:
    """
    A K koalíció értéke: a koalíció hibrid portfóliójának és a benchmarknak
    a kumulált hozamkülönbsége az adott időszakban (Davies–Laker-módszer).
    """

    mixed_actions = mix_actions(
        coalition,
        active_actions,
        benchmark_actions
    )

    mixed_return = return_function(
        mixed_actions,
        all_cells
    )

    benchmark_return = return_function(
        benchmark_actions,
        all_cells
    )

    common_index = (
        mixed_return.index
        .intersection(benchmark_return.index)
    )

    return cumulative_excess_return(
        mixed_return.loc[common_index],
        benchmark_return.loc[common_index]
    )

def compute_coalition_values(
    active_actions: AttributionActions,
    benchmark_actions: AttributionActions,
    decision_makers: List[str],
    all_cells: pd.MultiIndex
) -> Dict[FrozenSet[str], float]:

    values = {}

    for coalition_tuple in powerset(
        decision_makers
    ):

        coalition = frozenset(
            coalition_tuple
        )

        values[coalition] = (
            value_function_excess_return(
                coalition,
                active_actions,
                benchmark_actions,
                all_cells
            )
        )

    return values


def harsanyi_attributions(
    coalition_values: Dict[FrozenSet[str], float],
    decision_makers: List[str]
) -> Dict[FrozenSet[str], float]:

    psi = {}

    for coalition_tuple in powerset(
        decision_makers
    ):

        coalition = frozenset(
            coalition_tuple
        )

        if len(coalition) == 0:
            continue

        strict_subcoalition_sum = 0.0

        for sub_tuple in powerset(
            list(coalition)
        ):

            sub = frozenset(
                sub_tuple
            )

            if len(sub) == 0 or sub == coalition:
                continue

            strict_subcoalition_sum += (
                psi[sub]
            )

        psi[coalition] = (
            coalition_values[coalition]
            - strict_subcoalition_sum
        )

    return psi


def shapley_from_harsanyi(
    harsanyi: Dict[FrozenSet[str], float],
    decision_makers: List[str]
) -> pd.DataFrame:

    shapley = {
        dm: 0.0
        for dm in decision_makers
    }

    for coalition, value in harsanyi.items():

        share = value / len(
            coalition
        )

        for dm in coalition:
            shapley[dm] += share

    df = pd.DataFrame(
        {
            "decision_maker":
                list(shapley.keys()),
            "shapley_value":
                list(shapley.values())
        }
    ).set_index(
        "decision_maker"
    )

    df = df.sort_values(
        "shapley_value",
        ascending=False
    )

    df["shapley_value_pct_point"] = (
        df["shapley_value"] * 100.0
    )

    return df


def attribution_table(
    coalition_values: Dict[FrozenSet[str], float],
    harsanyi: Dict[FrozenSet[str], float]
) -> pd.DataFrame:

    rows = []

    for coalition, psi_value in (
        harsanyi.items()
    ):

        coalition_name = " + ".join(
            sorted(coalition)
        )

        rows.append(
            {
                "coalition": coalition_name,
                "size": len(coalition),
                "vS_cumulative_excess_return":
                    coalition_values[coalition],
                "harsanyi_attribution":
                    psi_value,
                "vS_pct_point":
                    coalition_values[coalition]
                    * 100.0,
                "harsanyi_attribution_pct_point":
                    psi_value * 100.0,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["size", "coalition"]
        )
        .reset_index(drop=True)
    )


def stock_weights_table(
    data: PortfolioData
) -> pd.DataFrame:

    table = data.metadata.copy()

    table["market_cap_usd"] = (
        data.market_caps
    )

    table["active_weight"] = (
        data.active_weights
    )

    table["benchmark_weight"] = (
        data.benchmark_weights
    )

    table["active_minus_benchmark"] = (
        table["active_weight"]
        - table["benchmark_weight"]
    )

    return table.sort_values(
        "benchmark_weight",
        ascending=False
    )


def region_sector_weight_tables(
    data: PortfolioData
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    active_region = compute_region_weights(
        data.active_weights,
        data.metadata
    )

    benchmark_region = compute_region_weights(
        data.benchmark_weights,
        data.metadata
    )

    region_table = pd.DataFrame(
        {
            "active_region_weight":
                active_region,
            "benchmark_region_weight":
                benchmark_region,
        }
    )

    region_table[
        "active_minus_benchmark"
    ] = (
        region_table[
            "active_region_weight"
        ]
        - region_table[
            "benchmark_region_weight"
        ]
    )

    active_sector = (
        compute_sector_weights_within_region(
            data.active_weights,
            data.metadata
        )
    )

    benchmark_sector = (
        compute_sector_weights_within_region(
            data.benchmark_weights,
            data.metadata
        )
    )

    sector_table = pd.DataFrame(
        {
            "active_sector_weight_within_region":
                active_sector,
            "benchmark_sector_weight_within_region":
                benchmark_sector,
        }
    )

    sector_table[
        "active_minus_benchmark"
    ] = (
        sector_table[
            "active_sector_weight_within_region"
        ]
        - sector_table[
            "benchmark_sector_weight_within_region"
        ]
    )

    return region_table, sector_table


def cell_composition_table(
    metadata: pd.DataFrame
) -> pd.DataFrame:

    rows = []

    for (
        region,
        sector
    ), group in metadata.groupby(
        ["region", "sector"]
    ):

        rows.append(
            {
                "region": region,
                "sector": sector,
                "n_stocks": len(group),
                "tickers": ", ".join(
                    group.index
                ),
                "companies": ", ".join(
                    group["company"]
                    .astype(str)
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["region", "sector"]
        )
    )


def data_quality_report(
    data: PortfolioData
) -> pd.DataFrame:

    return pd.DataFrame(
        [
            {
                "metric": "excel_input_path",
                "value": EXCEL_INPUT_PATH
            },
            {
                "metric": "price_sheet",
                "value": PRICE_SHEET
            },
            {
                "metric": "market_cap_sheet",
                "value": MARKET_CAP_SHEET
            },
            {
                "metric": "metadata_sheet",
                "value": METADATA_SHEET
            },
            {
                "metric": "start_date",
                "value":
                    str(
                        data.returns.index.min().date()
                    )
            },
            {
                "metric": "end_date",
                "value":
                    str(
                        data.returns.index.max().date()
                    )
            },
            {
                "metric": "n_monthly_observations",
                "value": len(data.returns)
            },
            {
                "metric": "n_stocks",
                "value": len(
                    data.returns.columns
                )
            },
            {
                "metric": "base_currency",
                "value": BASE_CURRENCY
            },
            {
                "metric":
                    "market_cap_reference_date_requested",
                "value":
                    MARKET_CAP_REFERENCE_DATE
            },
            {
                "metric":
                    "market_cap_reference_date_used",
                "value":
                    data.benchmark_reference_date
            },
            {
                "metric":
                    "missing_return_values_after_cleaning",
                "value":
                    int(
                        data.returns.isna()
                        .sum()
                        .sum()
                    )
            },
            {
                "metric":
                    "active_weight_sum",
                "value":
                    float(
                        data.active_weights.sum()
                    )
            },
            {
                "metric":
                    "benchmark_weight_sum",
                "value":
                    float(
                        data.benchmark_weights.sum()
                    )
            },
        ]
    )


def slice_data_by_period(
    data: PortfolioData,
    start_date: str,
    end_date: str,
    benchmark_reference_date: str
) -> PortfolioData:
    returns_period = data.returns.loc[start_date:end_date].copy()
    prices_period = data.prices.loc[returns_period.index].copy()
    if returns_period.empty:
        raise ValueError(f"Üres időszak: {start_date} - {end_date}")

    benchmark_weights, used_reference_date = build_benchmark_weights_from_market_caps(
        data.market_cap_history, list(data.returns.columns), benchmark_reference_date
    )

    return PortfolioData(
        prices=prices_period,
        returns=returns_period,
        active_weights=data.active_weights,
        benchmark_weights=benchmark_weights,
        market_caps=data.market_cap_history.loc[pd.to_datetime(used_reference_date)],
        market_cap_history=data.market_cap_history,
        metadata=data.metadata,
        benchmark_reference_date=used_reference_date
    )


def run_attribution_for_data(
    data: PortfolioData,
    period_name: str,
    start_date: str,
    end_date: str
) -> AttributionResult:

    active_actions = (
        build_attribution_actions(
            data.returns,
            data.active_weights,
            data.metadata
        )
    )

    benchmark_actions = (
        build_attribution_actions(
            data.returns,
            data.benchmark_weights,
            data.metadata
        )
    )

    all_cells = (
        benchmark_actions.cell_returns.columns
        .union(
            active_actions.cell_returns.columns
        )
    )

    summary = summarize_active_vs_benchmark(
        data
    )

    performance_ts = (
        build_performance_timeseries(
            data
        )
    )

    coalition_values = (
        compute_coalition_values(
            active_actions,
            benchmark_actions,
            DECISION_MAKERS,
            all_cells
        )
    )

    harsanyi = harsanyi_attributions(
        coalition_values,
        DECISION_MAKERS
    )

    attr_df = attribution_table(
        coalition_values,
        harsanyi
    )

    shapley_df = shapley_from_harsanyi(
        harsanyi,
        DECISION_MAKERS
    )

    total_excess_vN = coalition_values[
        frozenset(DECISION_MAKERS)
    ]

    total_harsanyi = sum(
        harsanyi.values()
    )

    efficiency_check = pd.DataFrame(
        [
            {
                "period": period_name,
                "start_date": start_date,
                "end_date": end_date,
                "benchmark_reference_date": data.benchmark_reference_date,
                "total_excess_vN":
                    total_excess_vN,
                "sum_harsanyi_attributions":
                    total_harsanyi,
                "difference":
                    total_excess_vN
                    - total_harsanyi,
                "total_excess_vN_pct_point":
                    total_excess_vN * 100.0,
                "sum_harsanyi_attributions_pct_point":
                    total_harsanyi * 100.0,
                "difference_pct_point":
                    (
                        total_excess_vN
                        - total_harsanyi
                    ) * 100.0,
                "value_function": "Davies-Laker: kumulalt hozamok kulonbsege",
                "independent_check_v_N":
                    cumulative_excess_return(
                        performance_ts["active_return"],
                        performance_ts["benchmark_return"]
                    ),
                "independent_check_difference":
                    total_excess_vN
                    - cumulative_excess_return(
                        performance_ts["active_return"],
                        performance_ts["benchmark_return"]
                    ),
            }
        ]
    )

    return AttributionResult(
        period_name,
        start_date,
        end_date,
        summary,
        attr_df,
        shapley_df,
        efficiency_check,
        performance_ts
    )


def run_period_attributions(
    data: PortfolioData
) -> Dict[str, AttributionResult]:

    periods = {
        "full_2016_2025":
            ("2016-01-01", "2025-12-31"),

        "pre_covid_2016_2019":
            ("2016-01-01", "2019-12-31"),

        "covid_2020_2021":
            ("2020-01-01", "2021-12-31"),

        "inflation_ai_2022_2025":
            ("2022-01-01", "2025-12-31"),
    }

    results = {}

    for period_name, (
        start,
        end
    ) in periods.items():

        period_data = slice_data_by_period(
            data,
            start,
            end,
            PERIOD_BENCHMARK_REFERENCE_DATES[period_name]
        )

        results[period_name] = (
            run_attribution_for_data(
                period_data,
                period_name,
                start,
                end
            )
        )

    return results


def combined_period_attribution_table(
    results: Dict[str, AttributionResult]
) -> pd.DataFrame:

    frames = []

    for period_name, result in results.items():

        temp = result.attribution.copy()
        temp["period"] = period_name
        frames.append(temp)

    combined = pd.concat(
        frames,
        ignore_index=True
    )

    pivot = combined.pivot_table(
        index="coalition",
        columns="period",
        values="harsanyi_attribution_pct_point",
        aggfunc="first"
    )

    return pivot.reset_index()


def combined_period_efficiency_table(
    results: Dict[str, AttributionResult]
) -> pd.DataFrame:

    return pd.concat(
        [
            result.efficiency_check
            for result in results.values()
        ],
        ignore_index=True
    )


def save_base_outputs(
    data: PortfolioData,
    output_dir: str
) -> None:

    stock_weights_table(
        data
    ).to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_stock_weights.csv"
        ),
        encoding="utf-8-sig"
    )

    data.market_caps.to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_market_caps_used.csv"
        ),
        encoding="utf-8-sig"
    )

    region_weights, sector_weights = (
        region_sector_weight_tables(
            data
        )
    )

    region_weights.to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_region_weights.csv"
        ),
        encoding="utf-8-sig"
    )

    sector_weights.to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_sector_weights_within_region.csv"
        ),
        encoding="utf-8-sig"
    )

    cell_composition_table(
        data.metadata
    ).to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_cell_composition.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    data_quality_report(
        data
    ).to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_data_quality_report.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )


def save_attribution_outputs(
    results: Dict[str, AttributionResult],
    output_dir: str
) -> None:

    for period_name, result in results.items():

        prefix = (
            f"{RUN_LABEL}_{period_name}"
        )

        result.summary.to_csv(
            os.path.join(
                output_dir,
                f"{prefix}_summary.csv"
            ),
            encoding="utf-8-sig"
        )

        result.attribution.to_csv(
            os.path.join(
                output_dir,
                f"{prefix}_harsanyi_attribution.csv"
            ),
            index=False,
            encoding="utf-8-sig"
        )

        result.shapley.to_csv(
            os.path.join(
                output_dir,
                f"{prefix}_shapley.csv"
            ),
            encoding="utf-8-sig"
        )

        result.efficiency_check.to_csv(
            os.path.join(
                output_dir,
                f"{prefix}_efficiency_check.csv"
            ),
            index=False,
            encoding="utf-8-sig"
        )

        result.performance_timeseries.to_csv(
            os.path.join(
                output_dir,
                f"{prefix}_performance_timeseries.csv"
            ),
            encoding="utf-8-sig"
        )

    combined_period_attribution_table(
        results
    ).to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_combined_period_harsanyi.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

    combined_period_efficiency_table(
        results
    ).to_csv(
        os.path.join(
            output_dir,
            f"{RUN_LABEL}_combined_efficiency_check.csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )


def save_excel_workbook(
    data: PortfolioData,
    results: Dict[str, AttributionResult],
    output_dir: str
) -> None:

    excel_path = os.path.join(
        output_dir,
        f"{RUN_LABEL}_results.xlsx"
    )

    region_weights, sector_weights = (
        region_sector_weight_tables(
            data
        )
    )

    with pd.ExcelWriter(
        excel_path,
        engine="openpyxl"
    ) as writer:

        stock_weights_table(
            data
        ).to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "stock_weights"
            )
        )

        data.market_caps.to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "market_caps"
            )
        )

        region_weights.to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "region_weights"
            )
        )

        sector_weights.to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "sector_weights"
            )
        )

        cell_composition_table(
            data.metadata
        ).to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "cell_composition"
            ),
            index=False
        )

        data_quality_report(
            data
        ).to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "data_quality"
            ),
            index=False
        )

        combined_period_attribution_table(
            results
        ).to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "combined_harsanyi"
            ),
            index=False
        )

        combined_period_efficiency_table(
            results
        ).to_excel(
            writer,
            sheet_name=safe_excel_sheet_name(
                "efficiency_check"
            ),
            index=False
        )


        for period_name, result in results.items():

            short_period = period_name.replace(
                "inflation_ai",
                "infl_ai"
            )

            result.summary.to_excel(
                writer,
                sheet_name=safe_excel_sheet_name(
                    f"{short_period}_summary"
                )
            )

            result.attribution.to_excel(
                writer,
                sheet_name=safe_excel_sheet_name(
                    f"{short_period}_attr"
                ),
                index=False
            )

            result.shapley.to_excel(
                writer,
                sheet_name=safe_excel_sheet_name(
                    f"{short_period}_shapley"
                )
            )

            result.efficiency_check.to_excel(
                writer,
                sheet_name=safe_excel_sheet_name(
                    f"{short_period}_eff"
                ),
                index=False
            )

            result.performance_timeseries.to_excel(
                writer,
                sheet_name=safe_excel_sheet_name(
                    f"{short_period}_ts"
                )
            )

    print(
        f"\nExcel output mentve ide: {excel_path}"
    )


def print_key_outputs(
    data: PortfolioData,
    results: Dict[str, AttributionResult]
) -> None:

    pd.set_option(
        "display.width",
        220
    )

    pd.set_option(
        "display.max_columns",
        40
    )

    pd.set_option(
        "display.max_rows",
        120
    )

    print("\n" + "=" * 110)
    print("ADATELLENŐRZÉS")
    print("=" * 110)

    print(
        f"Részvények száma: "
        f"{len(data.returns.columns)}"
    )

    print(
        f"Havi hozam-megfigyelések: "
        f"{len(data.returns)}"
    )

    print(
        f"Hozamadat kezdete: "
        f"{data.returns.index.min().date()}"
    )

    print(
        f"Hozamadat vége: "
        f"{data.returns.index.max().date()}"
    )

    print(
        f"Benchmark referencia-dátum: "
        f"{data.benchmark_reference_date}"
    )

    print(
        f"Aktív súlyok összege: "
        f"{data.active_weights.sum():.10f}"
    )

    print(
        f"Benchmark súlyok összege: "
        f"{data.benchmark_weights.sum():.10f}"
    )

    print(
        f"Hiányzó hozamértékek: "
        f"{data.returns.isna().sum().sum()}"
    )

    print("\nRégiók:")

    region_counts = pd.Series(
        {
            ticker:
                ASSET_METADATA[ticker]["region"]
            for ticker in data.returns.columns
        }
    ).value_counts()

    print(region_counts)

    print("\nSzektorok:")

    sector_counts = pd.Series(
        {
            ticker:
                ASSET_METADATA[ticker]["sector"]
            for ticker in data.returns.columns
        }
    ).value_counts()

    print(sector_counts)

    full_result = results[
        "full_2016_2025"
    ]

    print("\n" + "=" * 110)
    print("TELJES IDŐSZAK: TELJESÍTMÉNY")
    print("=" * 110)

    print(
        full_result.summary.round(6)
    )

    print("\n" + "=" * 110)
    print("TELJES IDŐSZAK: HARSÁNYI-ATTRIBÚCIÓ")
    print("=" * 110)

    print(
        full_result.attribution.round(6)
    )

    print("\n" + "=" * 110)
    print("TELJES IDŐSZAK: SHAPLEY-ÉRTÉKEK")
    print("=" * 110)

    print(
        full_result.shapley.round(6)
    )

    print("\n" + "=" * 110)
    print("RÉSZIDŐSZAKI HARSÁNYI-ATTRIBÚCIÓK")
    print("=" * 110)

    print(
        combined_period_attribution_table(
            results
        ).round(6)
    )

    print("\n" + "=" * 110)
    print("HATÉKONYSÁGI ELLENŐRZÉS")
    print("=" * 110)

    print(
        combined_period_efficiency_table(
            results
        ).round(10)
    )


def main() -> None:

    ensure_output_dir(
        OUTPUT_DIR
    )

    if DELETE_OLD_EXCEL_OUTPUTS:
        delete_old_excel_outputs(
            OUTPUT_DIR
        )

    print(
        "\nHarsányi-attribúciós modell indítása..."
    )

    data = load_portfolio_data()

    results = run_period_attributions(
        data
    )

    if CREATE_CSV_OUTPUTS:
        save_base_outputs(
            data,
            OUTPUT_DIR
        )

        save_attribution_outputs(
            results,
            OUTPUT_DIR
        )

    save_excel_workbook(
        data,
        results,
        OUTPUT_DIR
    )

    print_key_outputs(
        data,
        results
    )

    print("\n" + "=" * 110)
    print("FUTTATÁS KÉSZ")
    print("=" * 110)

    print(
        f"Kimeneti mappa: {OUTPUT_DIR}"
    )

    print(
        "A modell 100 részvényre, "
        "2016–2025 havi adatokra futott."
    )

    print(
        "\nMódszertani beállítások:"
    )

    print(
        "- Aktív portfólió: hierarchikus egyenlő súlyozás; azonos régiósúly, azon belül azonos szektorsúly, azon belül azonos részvénysúly."
    )

    print(
        "- Benchmark: időszakonként újrasúlyozott market-cap benchmark (2015-12-31 / 2019-12-31 / 2021-12-31)."
    )

    print(
        "- Döntéshozók: REGION, SECTOR, SELECTION."
    )

    print(
        "- Értékfüggvény: Davies–Laker-módszer (a hibrid portfólió és a benchmark kumulált hozamának különbsége)."
    )

    print(
        "- Market cap adatok: Bloomberg-alapú Excel."
    )

    print(
        "- Árfolyamadatok: Bloomberg-alapú Excel."
    )


if __name__ == "__main__":
    main()



Harsányi-attribúciós modell indítása...

Excel output mentve ide: outputs_harsanyi_100stocks_hierarchical_davies_laker\2016_2025_100stocks_hierarchical_active_marketcap_benchmark_davies_laker_results.xlsx

ADATELLENŐRZÉS
Részvények száma: 100
Havi hozam-megfigyelések: 120
Hozamadat kezdete: 2016-01-29
Hozamadat vége: 2025-12-31
Benchmark referencia-dátum: 2015-12-31
Aktív súlyok összege: 1.0000000000
Benchmark súlyok összege: 1.0000000000
Hiányzó hozamértékek: 0

Régiók:
North America    40
Europe           30
Asia             30
dtype: int64

Szektorok:
Utilities                 10
Health Care                9
Information Technology     9
Financials                 9
Energy                     9
Consumer Staples           9
Consumer Discretionary     9
Industrials                9
Communication Services     9
Materials                  9
Real Estate                9
dtype: int64

TELJES IDŐSZAK: TELJESÍTMÉNY
                                            cumulative_return  annualized_re